In [1]:
from analytics.db import execute,query

In [2]:
query('''SELECT 
    EXTRACT(YEAR FROM order_purchase_timestamp) AS order_year,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_year
ORDER BY order_year ASC;

      ''')

,order_year,total_orders
0,2016.0,329
1,2017.0,45101
2,2018.0,54011


> **Interpretation:** Profiling table row counts establishes the foundational scale and record volume for each entity in the Olist schema.

In [34]:
# Count the 8 schema entities/metrics
query("""
    SELECT 
        (SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'public') AS total_tables,
        (SELECT COUNT(*) FROM orders) AS total_orders,
        (SELECT COUNT(*) FROM customers) AS total_customers,
        (SELECT COUNT(*) FROM order_items) AS total_items,
        (SELECT COUNT(*) FROM products) AS total_products,
        (SELECT COUNT(*) FROM sellers) AS total_sellers,
        (SELECT COUNT(*) FROM order_payments) AS total_payments,
        (SELECT COUNT(*) FROM order_reviews) AS total_reviews
""")

,total_tables,total_orders,total_customers,total_items,total_products,total_sellers,total_payments,total_reviews
0,9,99441,99441,112650,32951,3095,103886,99224


In [12]:
query("""
    SELECT COUNT(*)                                       AS rows,
           COUNT(DISTINCT order_id)                       AS different_values,
           COUNT(*) FILTER (WHERE order_id IS NULL)       AS missing
    FROM orders
""")

,rows,different_values,missing
0,99441,99441,0


Tested `order_id` as the primary key. It is proven because the table has 99,441 rows and 99,441 unique values.

In [13]:
query("""
    SELECT COUNT(*) AS rows, 
           COUNT(DISTINCT (order_id, order_item_id)) AS different_pairs 
    FROM order_items
""")

,rows,different_pairs
0,112650,112650


Tested `order_id` and `order_item_id` together as a composite primary key. It is proven because the table has 112,650 rows and 112,650 unique pairs.

In [14]:
query("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state)) AS distinct_rows
    FROM geolocation
""")

,total_rows,distinct_rows
0,1000163,738332


Tested `geolocation_zip_code_prefix` as a primary key. It is not proven because zip codes repeat across multiple cities and states.

In [15]:
query("""
    SELECT COUNT(*) AS rows, 
           COUNT(DISTINCT (order_id, order_item_id,product_id , seller_id)) AS different_pairs 
    FROM order_items
""")

,rows,different_pairs
0,112650,112650


In [16]:
query("""
    SELECT COUNT(*)                                         AS rows,
           COUNT(DISTINCT customer_id)                      AS diff_customer_id,
           COUNT(DISTINCT customer_unique_id)               AS diff_unique_id,
           COUNT(*) FILTER (WHERE customer_id IS NULL)      AS missing_customer_id
    FROM customers
""")

,rows,diff_customer_id,diff_unique_id,missing_customer_id
0,99441,99441,96096,0


Tested `customer_id` as the primary key. It is proven because the table has 99,441 rows and 99,441 unique values.


In [17]:
query("""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT order_id) AS diff_orders,
           COUNT(DISTINCT (order_id, payment_sequential)) AS diff_pairs
    FROM order_payments
""")

,rows,diff_orders,diff_pairs
0,103886,99440,103886


Tested `order_id` and `payment_sequential` together as a composite primary key. It is proven because there are 103,886 rows and 103,886 unique pairs.

In [18]:
query("""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT review_id) AS diff_reviews,
           COUNT(DISTINCT (review_id, order_id)) AS diff_pairs
    FROM order_reviews
""")

,rows,diff_reviews,diff_pairs
0,99224,98410,99224


Tested `review_id` and `order_id` together as a composite primary key. It is proven because there are 99,224 rows and 99,224 unique pairs.

In [19]:
query("SELECT COUNT(*) AS rows, COUNT(DISTINCT product_id) AS diff FROM products")

,rows,diff
0,32951,32951


Tested `product_id` as the primary key. It is proven because the table has 32,951 rows and 32,951 unique values.

In [20]:
query("SELECT COUNT(*) AS rows, COUNT(DISTINCT seller_id) AS diff FROM sellers")

,rows,diff
0,3095,3095


Tested `seller_id` as the primary key. It is proven because the table has 3,095 rows and 3,095 unique values.

In [21]:
query("SELECT COUNT(*) AS rows, COUNT(DISTINCT product_category_name) AS diff FROM category_translation")

,rows,diff
0,71,71


Tested `product_category_name` as the primary key. It is proven because the table has 71 rows and 71 unique values.

In [22]:
query("""
    SELECT COUNT(*)
    FROM orders AS o
    WHERE NOT EXISTS (
        SELECT 1 FROM customers AS c
        WHERE c.customer_id = o.customer_id
    )
""")

,count
0,0


In [23]:
query("""
    SELECT 
        MIN(order_purchase_timestamp) AS first_order,
        MAX(order_purchase_timestamp) AS last_order,
        COUNT(DISTINCT order_status) AS unique_statuses
    FROM orders
""")

,first_order,last_order,unique_statuses
0,2016-09-04 21:15:19,2018-10-17 17:30:18,8


In [24]:
query("""
    SELECT order_status, COUNT(*) AS count
    FROM orders
    GROUP BY order_status
    ORDER BY count DESC
""")

,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [36]:
query("""
    SELECT 
        o.order_status,
        COUNT(o.order_id) AS orders_with_no_items
    FROM orders AS o
    WHERE NOT EXISTS (
        SELECT 1 FROM order_items AS i WHERE i.order_id = o.order_id
    )
    GROUP BY o.order_status
    ORDER BY orders_with_no_items DESC
""")

,order_status,orders_with_no_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [25]:
query("""
    SELECT 
        MIN(price) AS min_price,
        ROUND(CAST(MAX(price) AS numeric), 2) AS max_price,
        ROUND(CAST(AVG(price) AS numeric), 2) AS avg_price,
        MIN(freight_value) AS min_freight,
        ROUND(CAST(MAX(freight_value) AS numeric), 2) AS max_freight
    FROM order_items
""")

,min_price,max_price,avg_price,min_freight,max_freight
0,0.85,6735.0,120.65,0.0,409.68


In [26]:
query("""
    SELECT 
        COUNT(DISTINCT product_category_name) AS unique_categories,
        COUNT(*) AS total_products
    FROM products
""")

,unique_categories,total_products
0,73,32951


In [27]:
query("""
      SELECT review_id, COUNT(*) AS cnt, COUNT(DISTINCT order_id) AS distinct_orders, COUNT(DISTINCT review_score) AS distinct_scores
FROM order_reviews
GROUP BY review_id
HAVING COUNT(*) > 1
LIMIT 5;
      
      
      """)

,review_id,cnt,distinct_orders,distinct_scores
0,00130cbe1f9d422698c812ed8ded1919,2,2,1
1,0115633a9c298b6a98bcbe4eee75345f,2,2,1
2,0174caf0ee5964646040cd94e15ac95e,2,2,1
3,017808d29fd1f942d97e50184dfb4c13,2,2,1
4,0254bd905dc677a6078990aad3331a36,2,2,1


Repeated review IDs occur when the same customer (same customer_unique_id) submits a review that gets attached to multiple different orders over time, rather than multiple items within a single order.

In [28]:
query("""
      WITH zip_counts AS (
    SELECT geolocation_zip_code_prefix, COUNT(*) AS cnt, COUNT(DISTINCT geolocation_state) AS states_count
    FROM geolocation
    GROUP BY geolocation_zip_code_prefix
)
SELECT 
    MIN(cnt) AS min_count,
    MAX(cnt) AS max_count,
    SUM(CASE WHEN states_count > 1 THEN 1 ELSE 0 END) AS multi_state_zip_codes
FROM zip_counts;""")

,min_count,max_count,multi_state_zip_codes
0,1,1146,8


Zip code counts show variation from single entries to densely populated regions, with some zip codes spanning across multiple states. One row in this table represents a specific coordinate point and postal prefix combination rather than a unique geographic boundary.

In [35]:
query("""
    SELECT 
        geolocation_zip_code_prefix AS zip,
        COUNT(*) AS rows_count,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY geolocation_lat) AS median_lat,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY geolocation_lng) AS median_lng
    FROM geolocation
    GROUP BY geolocation_zip_code_prefix
    LIMIT 5
""")

,zip,rows_count,median_lat,median_lng
0,01001,26,-23.550381,-46.634027
1,01002,13,-23.548551,-46.635072
2,01003,17,-23.548977,-46.635313
3,01004,22,-23.549535,-46.634771
4,01005,25,-23.549612,-46.636532


In [29]:
query("""
      SELECT 
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customer_ids
FROM orders;
      
      """)

,total_orders,unique_customer_ids
0,99441,99441


Every order has its own unique customer_id (99,441 values for 99,441 orders), meaning customer_id represents a single order instance rather than a person. True individuals are tracked by customer_unique_id (96,096 values).

In [ ]:
query("""
    SELECT o.order_status, COUNT(*) AS orders_with_no_items
    FROM orders AS o
    WHERE NOT EXISTS (
        SELECT 1 FROM order_items AS i WHERE i.order_id = o.order_id
    )
    GROUP BY o.order_status
    ORDER BY orders_with_no_items DESC;
""")

,order_status,orders_with_no_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


Counted orders with no items grouped by each order_status to analyze which order statuses lack item records.

The geolocation table cannot be a strict foreign key because postal codes map to multiple coordinate points and states, resulting in a many-to-many relationship rather than a unique parent lookup.

In [31]:
query("""
      SELECT 
    (SELECT COUNT(*) FROM orders WHERE order_id NOT IN (SELECT order_id FROM order_items)) AS no_items,
    (SELECT COUNT(*) FROM orders WHERE order_id NOT IN (SELECT order_id FROM order_payments)) AS no_payment,
    (SELECT COUNT(*) FROM orders WHERE order_id NOT IN (SELECT order_id FROM order_reviews)) AS no_review;
         
      
      
      """)

,no_items,no_payment,no_review
0,775,1,768


Counted incomplete orders: orders with missing items, orders without payment records, and orders lacking customer reviews to quantify data completeness anomalies.

In [32]:
from analytics.profile import profile_table

df_products = profile_table("products")
display(df_products)

,column,type,nulls,null_pct,distinct,min,max
0,product_id,text,0,0.0,32951,00066f42aeeb9f3007548bb9d3f33c38,fffe9eeff12fcbd74a2f2b007dde0c58
1,product_category_name,text,610,1.9,73,agro_industria_e_comercio,utilidades_domesticas
2,product_name_lenght,integer,610,1.9,66,5,76
3,product_description_lenght,integer,610,1.9,2960,4,3992
4,product_photos_qty,integer,610,1.9,19,1,20
5,product_weight_g,integer,2,0.0,2204,0,40425
6,product_length_cm,integer,2,0.0,99,7,105
7,product_height_cm,integer,2,0.0,102,2,105
8,product_width_cm,integer,2,0.0,95,6,118


In [33]:
from analytics.profile import profile_table

tables = [
    "orders", "customers", "order_items", "order_payments", 
    "order_reviews", "products", "sellers", "category_translation", "geolocation"
]

for t in tables:
    print(f"=== Table: {t} ===")
    display(profile_table(t))

=== Table: orders ===


,column,type,nulls,null_pct,distinct,min,max
0,order_id,text,0,0.0,99441,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244
1,customer_id,text,0,0.0,99441,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99
2,order_status,text,0,0.0,8,approved,unavailable
3,order_purchase_timestamp,timestamp without time zone,0,0.0,98875,2016-09-04 21:15:19,2018-10-17 17:30:18
4,order_approved_at,timestamp without time zone,160,0.2,90733,2016-09-15 12:16:38,2018-09-03 17:40:06
5,order_delivered_carrier_date,timestamp without time zone,1783,1.8,81018,2016-10-08 10:34:01,2018-09-11 19:48:28
6,order_delivered_customer_date,timestamp without time zone,2965,3.0,95664,2016-10-11 13:46:32,2018-10-17 13:22:46
7,order_estimated_delivery_date,timestamp without time zone,0,0.0,459,2016-09-30 00:00:00,2018-11-12 00:00:00


=== Table: customers ===


,column,type,nulls,null_pct,distinct,min,max
0,customer_id,text,0,0.0,99441,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99
1,customer_unique_id,text,0,0.0,96096,0000366f3b9a7992bf8c76cfdf3221e2,ffffd2657e2aad2907e67c3e9daecbeb
2,customer_zip_code_prefix,text,0,0.0,14994,01003,99990
3,customer_city,text,0,0.0,4119,abadia dos dourados,zortea
4,customer_state,text,0,0.0,27,AC,TO


=== Table: order_items ===


,column,type,nulls,null_pct,distinct,min,max
0,order_id,text,0,0.0,98666,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244
1,order_item_id,integer,0,0.0,21,1,21
2,product_id,text,0,0.0,32951,00066f42aeeb9f3007548bb9d3f33c38,fffe9eeff12fcbd74a2f2b007dde0c58
3,seller_id,text,0,0.0,3095,0015a82c2db000af6aaaf3ae2ecb0532,ffff564a4f9085cd26170f4732393726
4,shipping_limit_date,timestamp without time zone,0,0.0,93318,2016-09-19 00:15:34,2020-04-09 22:35:08
5,price,numeric,0,0.0,5968,0.85,6735.00
6,freight_value,numeric,0,0.0,6999,0.00,409.68


... (8 outputs left out)

=== Table: category_translation ===


,column,type,nulls,null_pct,distinct,min,max
0,product_category_name,text,0,0.0,71,agro_industria_e_comercio,utilidades_domesticas
1,product_category_name_english,text,0,0.0,71,agro_industry_and_commerce,watches_gifts


=== Table: geolocation ===


,column,type,nulls,null_pct,distinct,min,max
0,geolocation_zip_code_prefix,text,0,0.0,19015,01001,99990
1,geolocation_lat,double precision,0,0.0,717372,-36.6053744107061,45.06593318269697
2,geolocation_lng,double precision,0,0.0,717615,-101.46676644931476,121.10539381057764
3,geolocation_city,text,0,0.0,8011,* cidade,zortéa
4,geolocation_state,text,0,0.0,27,AC,TO
